In [1]:
import os
os.chdir("../")
%pwd

'C:\\Users\\Sayantan Nandi\\Desktop\\mlops\\developer-burnout-project'

In [2]:
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any, List

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_ckpt: Path

    n_trials: int
    direction: str
    models: List[str]
    params: Dict[str, Any]

In [3]:
from src.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.utils.common import read_yaml, create_directories

In [4]:
class ConfigurationManager:
    def __init__(self,
                 config_path=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                ):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            train_data_path=Path(config.train_data_path),
            test_data_path=Path(config.test_data_path),
            model_ckpt=Path(config.model_ckpt),

            n_trials=params.optuna.n_trials,
            direction=params.optuna.direction,

            models=params.model_selection.models,

            params={
                k: v for k, v in params.items()
                if k not in ["model_selection", "optuna"]
            }
        )

        return model_trainer_config

In [5]:
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, classification_report

from src.logging import logger
from src.utils.common import create_directories
from src.utils.model_utils import tune_model

C:\Users\Sayantan Nandi\Desktop\mlops\developer-burnout-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class ModelTrainer:
    def __init__(self, config):
        self.config = config
        create_directories([self.config.root_dir])

    def train(self):
        logger.info("Starting model training process...")

        logger.info("Loading training and testing data...")

        train_df = pd.read_csv(self.config.train_data_path)
        test_df = pd.read_csv(self.config.test_data_path)

        X_train = train_df.drop("burnout_level", axis=1)
        y_train = train_df["burnout_level"]

        X_test = test_df.drop("burnout_level", axis=1)
        y_test = test_df["burnout_level"]

        logger.info("Data loaded successfully.")

        result = tune_model(X_train, y_train, self.config)

        best_model = result["model"]

        logger.info(f"Best model: {result['model_type']}")
        logger.info(f"Best params: {result['best_params']}")
        logger.info(f"Best CV score: {result['best_score']:.4f}")

        logger.info("Evaluating model on test data...")

        y_pred = best_model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred)

        logger.info(f"Test Accuracy: {accuracy:.4f}")
        logger.info(f"Classification Report:\n{report}")

        logger.info("Saving trained model...")

        joblib.dump(best_model, self.config.model_ckpt)

        logger.info(f"Model saved at {self.config.model_ckpt}")

        return accuracy

In [7]:
configurationManager = ConfigurationManager()
model_trainer_config = configurationManager.get_model_trainer_config()
model_trainer = ModelTrainer(config=model_trainer_config)
model_trainer.train()

[2026-04-16 21:01:55,275: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-16 21:01:55,280: INFO: common: yaml file: config\params.yaml loaded successfully]
[2026-04-16 21:01:55,281: INFO: common: created directory at: artifacts]
[2026-04-16 21:01:55,282: INFO: common: created directory at: artifacts/model_trainer]
[2026-04-16 21:01:55,283: INFO: common: created directory at: artifacts\model_trainer]
[2026-04-16 21:01:55,283: INFO: 4053691987: Starting model training process...]
[2026-04-16 21:01:55,283: INFO: 4053691987: Loading training and testing data...]
[2026-04-16 21:01:55,299: INFO: 4053691987: Data loaded successfully.]
[2026-04-16 21:01:55,301: INFO: model_utils: Starting Optuna hyperparameter tuning...]


[I 2026-04-16 21:01:55,303] A new study created in memory with name: no-name-833b2802-6b66-4548-b695-f8762a7f142d


[2026-04-16 21:01:55,306: INFO: model_utils: Initializing model: random_forest]
[2026-04-16 21:01:57,410: INFO: model_utils: Trial 0 completed | Model: random_forest | Score: 0.9914]


[I 2026-04-16 21:01:57,411] Trial 0 finished with value: 0.9914351609705555 and parameters: {'model': 'random_forest', 'n_estimators': 143, 'max_depth': 14, 'min_samples_split': 7, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.9914351609705555.


[2026-04-16 21:01:57,412: INFO: model_utils: Initializing model: logistic_regression]
[2026-04-16 21:01:57,526: INFO: model_utils: Trial 1 completed | Model: logistic_regression | Score: 0.9650]


...

[I 2026-04-16 21:04:23,094] Trial 73 finished with value: 0.9914351609705555 and parameters: {'model': 'random_forest', 'n_estimators': 163, 'max_depth': 19, 'min_samples_split': 3, 'min_samples_leaf': 3}. Best is trial 16 with value: 0.9917996257386845.


[2026-04-16 21:04:23,102: INFO: model_utils: Initializing model: random_forest]
[2026-04-16 21:04:25,598: INFO: model_utils: Trial 74 completed | Model: random_forest | Score: 0.9916]


[I 2026-04-16 21:04:25,599] Trial 74 finished with value: 0.9916174763762073 and parameters: {'model': 'random_forest', 'n_estimators': 173, 'max_depth': 10, 'min_samples_split': 6, 'min_samples_leaf': 2}. Best is trial 16 with value: 0.9917996257386845.


[2026-04-16 21:04:25,600: INFO: model_utils: Best model found: random_forest]
[2026-04-16 21:04:25,601: INFO: model_utils: Best parameters: {'n_estimators': 140, 'max_depth': 16, 'min_samples_split': 4, 'min_samples_leaf': 1}]
[2026-04-16 21:04:25,602: INFO: model_utils: Best CV score: 0.9918]
[2026-04-16 21:04:25,603: INFO: model_utils: Initializing model: random_forest]
[2026-04-16 21:04:25,603: INFO: model_utils: Training best model on full dataset]
[2026-04-16 21:04:26,098: INFO: model_utils: Model training completed]
[2026-04-16 21:04:26,099: INFO: 4053691987: Best model: random_forest]
[2026-04-16 21:04:26,100: INFO: 4053691987: Best params: {'n_estimators': 140, 'max_depth': 16, 'min_samples_split': 4, 'min_samples_leaf': 1}]
[2026-04-16 21:04:26,101: INFO: 4053691987: Best CV score: 0.9918]
[2026-04-16 21:04:26,101: INFO: 4053691987: Evaluating model on test data...]
[2026-04-16 21:04:26,124: INFO: 4053691987: Test Accuracy: 0.9920]
[2026-04-16 21:04:26,125: INFO: 4053691987: C

0.9919825072886297